# AG1 Audience Request — v2 (Expanded)

Updated segments per audience to increase scale.

**Changes from v1:**
- **Clean Living Supplement Seeker**: added Chemical-Free Seekers, Doctor-Recommended Solutions, Detox & Cleanse Obsession
- **High-Activity Daily Performer**: added High-Intensity & Bodybuilding, CrossFit & Functional Training, Running & Endurance, Data-Driven Wellness & At-Home Testing
- **Category-Committed Supplement Buyer**: added Cognitive Performance & Nootropics, Sleep Optimization Interest, Ingestible Beauty & Collagen
- **Gut-Forward Longevity Buyer**: added Joint Pain & Inflammation Relief Interest, Stress & Anxiety Relief Interest, Sleep Optimization Interest
- Output prefix updated to: `shopper360/spectrum/ag1_20260429`

## Audience Sizing

In [ ]:
import pandas as pd
import boto3
import io
import sys

AUDIENCES = {
    "The Clean Living Supplement Seeker": [
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/NonDiscountHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Clinical_Research-Driven_Wellness_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Clean_&_Non-Toxic_Obsession_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Organic_Lifestyle_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Chemical-Free_Seekers_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Doctor-Recommended_Solutions_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Detox_&_Cleanse_Obsession_L730D.tsv",
    ],
    "The High-Activity Daily Performer": [
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/AvidHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Athletic_Performance_&_Recovery_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Biohacking_&_Quantified_Self_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/High-Intensity_&_Bodybuilding_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/CrossFit_&_Functional_Training_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Running_&_Endurance_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Data-Driven_Wellness_&_At-Home_Testing_L730D.tsv",
    ],
    "The Category-Committed Supplement Buyer": [
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/AvidHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/HighLTVHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/20260316/SubscriptionHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/20260316/HighAOVHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Cognitive_Performance_&_Nootropics_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Sleep_Optimization_Interest_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Ingestible_Beauty_&_Collagen_L730D.tsv",
    ],
    "The Gut-Forward Longevity Buyer": [
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/NonDiscountHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Gut_Health_&_Digestion_Interest_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Functional_Mushroom_&_Adaptogen_Users_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Longevity_&_Anti-Aging_Science_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Joint_Pain_&_Inflammation_Relief_Interest_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Stress_&_Anxiety_Relief_Interest_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Sleep_Optimization_Interest_L730D.tsv",
    ],
}

DEDUP_KEYS = ["address", "zip"]


def normalize_schema(df):
    df.columns = df.columns.str.lower()
    if "address1" in df.columns and "address" not in df.columns:
        df = df.rename(columns={"address1": "address"})
    if "segment_name" in df.columns:
        df = df.drop(columns=["segment_name"])
    return df


def normalize_address(df):
    if "address" in df.columns:
        df["address"] = (
            df["address"]
            .str.strip()
            .str.lower()
            .str.replace(r"\bstreet\b", "st", regex=True)
            .str.replace(r"\bavenue\b", "ave", regex=True)
            .str.replace(r"\bdrive\b", "dr", regex=True)
            .str.replace(r"\broad\b", "rd", regex=True)
            .str.replace(r"\bboulevard\b", "blvd", regex=True)
            .str.replace(r"\blane\b", "ln", regex=True)
            .str.replace(r"\bcourt\b", "ct", regex=True)
            .str.replace(r"\s+", " ", regex=True)
        )
    if "zip" in df.columns:
        df["zip"] = df["zip"].astype(str).str.strip().str.zfill(5).str[:5]
    return df


def read_s3_tsv(s3_path, s3_client, cache):
    if s3_path in cache:
        print(f"    Reusing: {s3_path.split('/')[-1]}")
        return cache[s3_path]
    bucket, key = s3_path.replace("s3://", "").split("/", 1)
    filename = key.split("/")[-1]
    print(f"    Reading: {filename}")
    try:
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        df = pd.read_csv(io.BytesIO(obj["Body"].read()), sep="\t", low_memory=False)
        df = normalize_schema(df)
        print(f"      -> {len(df):,} rows")
        cache[s3_path] = df
        return df
    except Exception as e:
        print(f"      ERROR: {e}")
        return None

def size_audience(name, paths, s3_client, cache):
    print(f"\n  [{name}]")
    dfs = []
    for path in paths:
        df = read_s3_tsv(path, s3_client, cache)
        if df is None:
            continue
        missing = [k for k in DEDUP_KEYS if k not in df.columns]
        if missing:
            print(f"      WARNING: missing columns {missing} — skipping")
            continue
        dfs.append(df[[k for k in DEDUP_KEYS if k in df.columns]])
    if not dfs:
        return 0
    combined = pd.concat(dfs, ignore_index=True)
    combined = normalize_address(combined)
    combined = combined.dropna(subset=DEDUP_KEYS)
    deduped = combined.drop_duplicates(subset=DEDUP_KEYS)
    return len(deduped)


def main():
    print("Initializing S3 client...\n")
    print("Loading and sizing audiences...\n")
    s3 = boto3.client("s3")
    cache = {}
    results = {}

    for name, paths in AUDIENCES.items():
        count = size_audience(name, paths, s3, cache)
        results[name] = count

    print("\n=======================================================")
    print("AG1 — AUDIENCE SIZING v2 (Spectrum/Charter)")
    print("=======================================================")
    for name, count in results.items():
        print(f"  {name}")
        print(f"    Deduped households: {count:,}\n")
    print("=======================================================")


if __name__ == "__main__":
    main()


## Build & Upload to S3

In [ ]:
import pandas as pd
import boto3
import io

AUDIENCES = {
    "The Clean Living Supplement Seeker": [
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/NonDiscountHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Clinical_Research-Driven_Wellness_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Clean_&_Non-Toxic_Obsession_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Organic_Lifestyle_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Chemical-Free_Seekers_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Doctor-Recommended_Solutions_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Detox_&_Cleanse_Obsession_L730D.tsv",
    ],
    "The High-Activity Daily Performer": [
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/AvidHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Athletic_Performance_&_Recovery_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Biohacking_&_Quantified_Self_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/High-Intensity_&_Bodybuilding_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/CrossFit_&_Functional_Training_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Running_&_Endurance_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Data-Driven_Wellness_&_At-Home_Testing_L730D.tsv",
    ],
    "The Category-Committed Supplement Buyer": [
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/AvidHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/HighLTVHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/20260316/SubscriptionHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/20260316/HighAOVHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Cognitive_Performance_&_Nootropics_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Sleep_Optimization_Interest_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Ingestible_Beauty_&_Collagen_L730D.tsv",
    ],
    "The Gut-Forward Longevity Buyer": [
        "s3://proxima-science-infrastructure-prod/shopper360/20260413/NonDiscountHealthShopperAudience.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Gut_Health_&_Digestion_Interest_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Functional_Mushroom_&_Adaptogen_Users_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Longevity_&_Anti-Aging_Science_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Joint_Pain_&_Inflammation_Relief_Interest_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Stress_&_Anxiety_Relief_Interest_L730D.tsv",
        "s3://proxima-science-infrastructure-prod/shopper360/brandtag_audiences/20260417/TSV/Sleep_Optimization_Interest_L730D.tsv",
    ],
}

DEDUP_KEYS = ["address", "zip"]


def normalize_schema(df):
    df.columns = df.columns.str.lower()
    if "address1" in df.columns and "address" not in df.columns:
        df = df.rename(columns={"address1": "address"})
    if "segment_name" in df.columns:
        df = df.drop(columns=["segment_name"])
    return df


def normalize_address(df):
    if "address" in df.columns:
        df["address"] = (
            df["address"]
            .str.strip()
            .str.lower()
            .str.replace(r"\bstreet\b", "st", regex=True)
            .str.replace(r"\bavenue\b", "ave", regex=True)
            .str.replace(r"\bdrive\b", "dr", regex=True)
            .str.replace(r"\broad\b", "rd", regex=True)
            .str.replace(r"\bboulevard\b", "blvd", regex=True)
            .str.replace(r"\blane\b", "ln", regex=True)
            .str.replace(r"\bcourt\b", "ct", regex=True)
            .str.replace(r"\s+", " ", regex=True)
        )
    if "zip" in df.columns:
        df["zip"] = df["zip"].astype(str).str.strip().str.zfill(5).str[:5]
    return df


def read_s3_tsv(s3_path, s3_client, cache):
    if s3_path in cache:
        print(f"    Reusing: {s3_path.split('/')[-1]}")
        return cache[s3_path]
    bucket, key = s3_path.replace("s3://", "").split("/", 1)
    filename = key.split("/")[-1]
    print(f"    Reading: {filename}")
    try:
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        df = pd.read_csv(io.BytesIO(obj["Body"].read()), sep="\t", low_memory=False)
        df = normalize_schema(df)
        print(f"      -> {len(df):,} rows")
        cache[s3_path] = df
        return df
    except Exception as e:
        print(f"      ERROR: {e}")
        return None

OUTPUT_COLS = ["address", "city", "state", "zip"]
DEDUP_KEYS = ["address", "zip"]
OUTPUT_BUCKET = "proxima-science-infrastructure-prod"
OUTPUT_PREFIX = "shopper360/spectrum/ag1_20260429"

def build_audience(name, paths, s3_client, cache):
    print(f"\n  [{name}]")
    dfs = []
    for path in paths:
        df = read_s3_tsv(path, s3_client, cache)
        if df is None:
            continue
        missing = [k for k in DEDUP_KEYS if k not in df.columns]
        if missing:
            print(f"      WARNING: missing dedup columns {missing} — skipping")
            continue
        cols = [c for c in OUTPUT_COLS if c in df.columns]
        dfs.append(df[cols])
    if not dfs:
        print(f"      No valid files for this audience.")
        return None
    combined = pd.concat(dfs, ignore_index=True)
    for col in OUTPUT_COLS:
        if col not in combined.columns:
            combined[col] = None
    combined = combined[OUTPUT_COLS]
    combined = normalize_address(combined)
    combined = combined.dropna(subset=DEDUP_KEYS)
    deduped = combined.drop_duplicates(subset=DEDUP_KEYS)
    return deduped


def upload_to_s3(df, name, s3_client):
    filename = name.replace(" ", "_").replace("/", "-").replace("&", "and") + ".csv"
    key = f"{OUTPUT_PREFIX}/{filename}"
    buf = io.BytesIO()
    df.to_csv(buf, index=False)
    buf.seek(0)
    s3_client.put_object(Bucket=OUTPUT_BUCKET, Key=key, Body=buf.getvalue())
    print(f"      Uploaded {len(df):,} records -> s3://{OUTPUT_BUCKET}/{key}")


def main():
    print("Initializing S3 client...")
    s3 = boto3.client("s3")
    cache = {}
    print(f"\nOutput: s3://{OUTPUT_BUCKET}/{OUTPUT_PREFIX}/")
    print("\nBuilding audiences...\n")
    for name, paths in AUDIENCES.items():
        deduped = build_audience(name, paths, s3, cache)
        if deduped is None:
            continue
        upload_to_s3(deduped, name, s3)
    print("\nDone.")


if __name__ == "__main__":
    main()
